# PM2.5 Prediction Model

**Input:** `df_model_monthly.csv` built in `features.ipynb`.

| Model | Algorithm | Features                                              | Goal |
|---|---|-------------------------------------------------------|---|
| **A** | RF + XGBoost | environmental, spatial, contextual variables          | Policy story: what can municipalities act on? |
| **B** | RF | PM2.5 lags, rolling features, lagged pollutants, weather, context | Accuracy benchmark |
| **C** | Ridge | Same as A                                             | Linear baseline with signed coefficients |

**Two splits:** time (train past / test future) and spatial (train on some stations / test on unseen stations).

**Primary outputs:** files in `model_output/` consumed by `policy_translation.ipynb`.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.base import clone
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
print("Libraries loaded")


Libraries loaded


## Load final modeling dataset

This CSV was created in `features.ipynb` and already contains:
- date
- station identifier (`eoi_code`)
- weather/context variables
- lag features
- rolling features

For now, we only use the columns needed for Model A.

In [2]:
df = pd.read_csv("df_model_monthly.csv", parse_dates=["date"])

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Stations:", df["eoi_code"].nunique())

display(df.head())

Shape: (3986, 33)
Date range: 2020-01-01 00:00:00 to 2025-11-01 00:00:00
Stations: 144


,eoi_code,date,Year,Month,Season,PM2_5,PM10,NO2,O3,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,PM10_lag1,NO2_lag1,O3_lag1,Temp_Mean,Wind_Speed,Precipitation,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density,month_sin,month_cos
0,IT0459A,2020-04-01,2020,4,Spring,8.32,21.4,24.4,54.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.490000,10.310000,44.6,NaN,NaN,NaN,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,8.660254e-01,-0.500000
1,IT0459A,2020-05-01,2020,5,Spring,6.41,19.7,25.1,56.7,8.32,NaN,NaN,NaN,NaN,21.4,24.4,54.1,18.061290,10.125806,62.0,13.490000,10.310000,44.6,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,5.000000e-01,-0.866025
2,IT0459A,2020-06-01,2020,6,Summer,8.42,21.6,23.9,61.4,6.41,8.32,NaN,NaN,NaN,19.7,25.1,56.7,21.836667,9.896667,59.2,18.061290,10.125806,62.0,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,1.224647e-16,-1.000000
3,IT0459A,2020-07-01,2020,7,Summer,8.13,20.4,20.1,56.5,8.42,6.41,8.32,7.716667,1.132711,21.6,23.9,61.4,24.532258,9.580645,28.6,21.836667,9.896667,59.2,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-5.000000e-01,-0.866025
4,IT0459A,2020-08-01,2020,8,Summer,7.74,20.0,27.2,53.0,8.13,8.42,6.41,7.653333,1.086477,20.4,20.1,56.5,25.606452,10.080645,65.2,24.532258,9.580645,28.6,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-8.660254e-01,-0.500000


In [3]:
# Ensuring target exists, date is datetime, station id exists, PM2.5 has non-missing values
print("Target missing:", df["PM2_5"].isna().sum())
print("Date dtype:", df["date"].dtype)
print("Station id missing:", df["eoi_code"].isna().sum())

display(df[["date", "eoi_code", "PM2_5"]].head())

Target missing: 0
Date dtype: datetime64[us]
Station id missing: 0


,date,eoi_code,PM2_5
0,2020-04-01,IT0459A,8.32
1,2020-05-01,IT0459A,6.41
2,2020-06-01,IT0459A,8.42
3,2020-07-01,IT0459A,8.13
4,2020-08-01,IT0459A,7.74


## Modeling dataset and splits

Before building individual models, we create one shared modeling dataset and one shared split setup.

This keeps Models A, B, and C fully comparable:
- same rows
- same target
- same station grouping
- same time split
- same spatial split

Filtering only on PM2.5 history features required by the forecasting setup.
We do not filter on pollutant lags like O3, because those reflect real sensor availability and would remove too many observations.

In [4]:
required_history = [
    "PM2_5_lag1",
    "PM2_5_lag2",
    "PM2_5_lag3",
    "PM2_5_roll3_mean",
    "PM2_5_roll3_std"
]

df_model = df.dropna(subset=required_history).copy()

print("Rows before filtering:", len(df))
print("Rows after common history filter:", len(df_model))
print("Stations after filter:", df_model["eoi_code"].nunique())

display(df_model.head())

Rows before filtering: 3986
Rows after common history filter: 3554
Stations after filter: 114


,eoi_code,date,Year,Month,Season,PM2_5,PM10,NO2,O3,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,PM10_lag1,NO2_lag1,O3_lag1,Temp_Mean,Wind_Speed,Precipitation,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density,month_sin,month_cos
3,IT0459A,2020-07-01,2020,7,Summer,8.13,20.4,20.1,56.5,8.42,6.41,8.32,7.716667,1.132711,21.6,23.9,61.4,24.532258,9.580645,28.6,21.836667,9.896667,59.2,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-0.500000,-8.660254e-01
4,IT0459A,2020-08-01,2020,8,Summer,7.74,20.0,27.2,53.0,8.13,8.42,6.41,7.653333,1.086477,20.4,20.1,56.5,25.606452,10.080645,65.2,24.532258,9.580645,28.6,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-0.866025,-5.000000e-01
5,IT0459A,2020-09-01,2020,9,Autumn,7.20,16.9,28.6,35.6,7.74,8.13,8.42,8.096667,0.341223,20.0,27.2,53.0,20.910000,9.833333,57.8,25.606452,10.080645,65.2,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-1.000000,-1.836970e-16
6,IT0459A,2020-10-01,2020,10,Autumn,24.90,36.4,29.3,26.0,7.20,7.74,8.13,7.690000,0.467012,16.9,28.6,35.6,14.580645,8.735484,52.8,20.910000,9.833333,57.8,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-0.866025,5.000000e-01
7,IT0459A,2020-11-01,2020,11,Autumn,18.40,26.5,26.3,29.1,24.90,7.20,7.74,13.280000,10.066837,36.4,29.3,26.0,11.090000,9.396667,48.7,14.580645,8.735484,52.8,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,-0.500000,8.660254e-01


## Target and grouping variable

These objects will be reused by all models:
- `y` = PM2.5 target
- `groups` = station identifier for spatial split

In [5]:
target_col = "PM2_5"
y = df_model[target_col].copy()
groups = df_model["eoi_code"].copy()

print("Filtered modeling shape:", df_model.shape)
print("Target shape:", y.shape)
print("Unique stations:", groups.nunique())

Filtered modeling shape: (3554, 33)
Target shape: (3554,)
Unique stations: 114


## Common train/test splits

We define the split and reuse it for all models.

### Time split
Train on earlier months, test on later months

In [6]:
# Time split
cutoff_date = df_model["date"].quantile(0.8)
time_train_mask = df_model["date"] <= cutoff_date
time_test_mask = df_model["date"] > cutoff_date

print("Cutoff:", cutoff_date.date())
print("Train/test:", time_train_mask.sum(), time_test_mask.sum())

Cutoff: 2025-04-01
Train/test: 2885 669


### Spatial split
Train on some stations, test on unseen stations

In [7]:
# Spatial split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(df_model, y, groups=groups))

print("Spatial train/test rows:", len(tr_idx), len(te_idx))
print("Train stations:", df_model.iloc[tr_idx]["eoi_code"].nunique())
print("Test stations:", df_model.iloc[te_idx]["eoi_code"].nunique())

Spatial train/test rows: 2831 723
Train stations: 91
Test stations: 23


## Model A  (Random Forest & XGBoost)

In this section, we build Model A, an explanatory model designed to understand how environmental, spatial, and contextual factors influence PM2.5 levels.

Unlike predictive models that rely on pollutant inputs, Model A uses only non-pollutant features, allowing us to interpret the underlying drivers of air pollution.


### Modeling Approach

We train two models:

- Random Forest (RF)
- XGBoost (XGB)

Both models are evaluated under two scenarios:

- Time split - forecasting future observations
- Spatial split - generalizing to unseen locations

---

### Model Selection

The final model (Champion A) is selected based on:

> Lowest average RMSE across both time and spatial splits

This ensures the chosen model performs well both:
- temporally (future prediction)
- spatially (new locations)

---

### Final Model

The selected model is retrained on the full dataset to:

- maximize available information
- enable interpretation via SHAP
- support feature importance analysis

In [8]:
# Model A feature set
# Keep this list simple and explicit.
# If a column name does not exist in df_model, it will be skipped safely.

candidate_features_a = [
    # Seasonality
    "Season",
    "month_sin",
    "month_cos",

    # Weather
    "Temp_Mean",
    "Wind_Speed",
    "Precipitation",

    # Lagged weather (still okay for explanatory model)
    "Temp_Mean_lag1",
    "Wind_Speed_lag1",
    "Precipitation_lag1",

    # Spatial
    "Altitude",
    "Latitude",
    "Longitude",
    "Green_Ratio",
    "Population_Density",

    # Contextual / station metadata
    "Station_Type",
    "Station_Area",
]

features_a = [c for c in candidate_features_a if c in df_model.columns]

print("Model A features:", len(features_a))
print(features_a)


Model A features: 16
['Season', 'month_sin', 'month_cos', 'Temp_Mean', 'Wind_Speed', 'Precipitation', 'Temp_Mean_lag1', 'Wind_Speed_lag1', 'Precipitation_lag1', 'Altitude', 'Latitude', 'Longitude', 'Green_Ratio', 'Population_Density', 'Station_Type', 'Station_Area']


In [9]:
def make_preprocessor(feature_list):
    num_cols = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols = [c for c in feature_list if c not in num_cols]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])

def reg_metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2":   r2_score(y_true, y_pred),
    }

pre_a = make_preprocessor(features_a)
print("Preprocessors and metrics helper ready")

Preprocessors and metrics helper ready


### Build Model A feature matrix

We now create the feature matrix for Model A only.

Important:
- `y` is already shared
- split masks and indices are already shared
- only `X_a` is model-specific

In [10]:
X_a = df_model[features_a].copy()

print("Model A X shape:", X_a.shape)
display(X_a.head())

# y, groups, time_train_mask, time_test_mask, tr_idx, te_idx
# were already created in the setup section above.

Model A X shape: (3554, 16)


,Season,month_sin,month_cos,Temp_Mean,Wind_Speed,Precipitation,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1,Altitude,Latitude,Longitude,Green_Ratio,Population_Density,Station_Type,Station_Area
3,Summer,-0.500000,-8.660254e-01,24.532258,9.580645,28.6,21.836667,9.896667,59.2,15.0,43.5989,13.3419,0.813059,788.0,Background,Suburban
4,Summer,-0.866025,-5.000000e-01,25.606452,10.080645,65.2,24.532258,9.580645,28.6,15.0,43.5989,13.3419,0.813059,788.0,Background,Suburban
5,Autumn,-1.000000,-1.836970e-16,20.910000,9.833333,57.8,25.606452,10.080645,65.2,15.0,43.5989,13.3419,0.813059,788.0,Background,Suburban
6,Autumn,-0.866025,5.000000e-01,14.580645,8.735484,52.8,20.910000,9.833333,57.8,15.0,43.5989,13.3419,0.813059,788.0,Background,Suburban
7,Autumn,-0.500000,8.660254e-01,11.090000,9.396667,48.7,14.580645,8.735484,52.8,15.0,43.5989,13.3419,0.813059,788.0,Background,Suburban


### Model A — Time split

The dataset is divided based on the date:
- the model is trained on **earlier months**
- the model is tested on **later months**

This simulates a realistic forecasting scenario, where we use past data to predict future air pollution levels.

We train both:
- Random Forest (RF)
- XGBoost (XGB)

and compare their performance using MAE, RMSE, and R².

In [11]:
# Time split data
X_train_a_t = X_a.loc[time_train_mask] # Features for training(past data)
X_test_a_t = X_a.loc[time_test_mask] # Features for testing (future data)
y_train_t = y.loc[time_train_mask] # values for training
y_test_t = y.loc[time_test_mask] # values for testing

# RF - time split
# Build Pipeline: Preprocessig + RF

rf_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])
# Train model on past data
rf_a_t.fit(X_train_a_t, y_train_t)
pred_rf_a_t = rf_a_t.predict(X_test_a_t) #Predict PM2.5 on future data
m_rf_a_t = reg_metrics(y_test_t, pred_rf_a_t) #Evaluate predictions using MAE, RMSE, and R²
print("RF time split:", m_rf_a_t)

# XGBoost — time split
# Same process as RF
xgb_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_t.fit(X_train_a_t, y_train_t)
pred_xgb_a_t = xgb_a_t.predict(X_test_a_t)
m_xgb_a_t = reg_metrics(y_test_t, pred_xgb_a_t)

print("XGB time split:", m_xgb_a_t)

RF time split: {'MAE': 4.384798208136111, 'RMSE': 10.467598678079142, 'R2': 0.28906097480565107}
XGB time split: {'MAE': 4.288854528461907, 'RMSE': 10.397701191678784, 'R2': 0.29852387840636996}


### Model A — Spatial split

Instead of splitting the data by time, we split it by **monitoring stations** (`eoi_code`).

- the model is trained on data from a subset of stations
- the model is tested on **completely unseen stations**

We train both:
- Random Forest(RF)
- XGBoost(XGB)

In [12]:
# Spatial split data
# Split the dataset using indices created earlier (GroupShuffleSplit)
# Ensures that train and test contain DIFFERENT stations

X_train_a_s = X_a.iloc[tr_idx] # Features for training (some stations)
X_test_a_s = X_a.iloc[te_idx] # Features for testing (unseen stations)
y_train_s = y.iloc[tr_idx] #
y_test_s = y.iloc[te_idx]

# RF — spatial split
rf_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])
rf_a_s.fit(X_train_a_s, y_train_s)
pred_rf_a_s = rf_a_s.predict(X_test_a_s)
m_rf_a_s = reg_metrics(y_test_s, pred_rf_a_s)
print("RF spatial split:", m_rf_a_s)

# XGBoost — spatial split
xgb_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_s.fit(X_train_a_s, y_train_s)
pred_xgb_a_s = xgb_a_s.predict(X_test_a_s)
m_xgb_a_s = reg_metrics(y_test_s, pred_xgb_a_s)
print("XGB spatial split:", m_xgb_a_s)


RF spatial split: {'MAE': 3.858498905606994, 'RMSE': 6.0016643300137895, 'R2': 0.4744012671139043}
XGB spatial split: {'MAE': 4.001915914226864, 'RMSE': 6.117697449954574, 'R2': 0.45388149064091055}


### Model A — Model Selection

Pick the model with the lowest average RMSE across both splits.

In [13]:
# Champion selection by lowest avg RMSE across both splits
avg_rmse_rf = (m_rf_a_t["RMSE"] + m_rf_a_s["RMSE"]) / 2
avg_rmse_xgb = (m_xgb_a_t["RMSE"] + m_xgb_a_s["RMSE"]) / 2

print(f"RF avg RMSE: {avg_rmse_rf:.4f}")
print(f"XGB avg RMSE: {avg_rmse_xgb:.4f}")

if avg_rmse_rf <= avg_rmse_xgb:
    champion_a_name = "RF"
    pred_champ_a_t = pred_rf_a_t
    pred_champ_a_s = pred_rf_a_s
    m_champ_a_t = m_rf_a_t
    m_champ_a_s = m_rf_a_s
else:
    champion_a_name = "XGBoost"
    pred_champ_a_t = pred_xgb_a_t
    pred_champ_a_s = pred_xgb_a_s
    m_champ_a_t = m_xgb_a_t
    m_champ_a_s = m_xgb_a_s

print(f"\nChampion A: {champion_a_name}")
print(f"Time split - RMSE={m_champ_a_t['RMSE']:.4f}  MAE={m_champ_a_t['MAE']:.4f}  R2={m_champ_a_t['R2']:.4f}")
print(f"Spatial split - RMSE={m_champ_a_s['RMSE']:.4f}  MAE={m_champ_a_s['MAE']:.4f}  R2={m_champ_a_s['R2']:.4f}")

# Retrain champion on FULL dataset (for SHAP / feature importance)
if champion_a_name == "RF":
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )),
    ])
else:
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )),
    ])

champ_a_full.fit(X_a, y)
print(f"\nchamp_a_full retrained on full dataset ({len(X_a)} rows).")


RF avg RMSE: 8.2346
XGB avg RMSE: 8.2577

Champion A: RF
Time split - RMSE=10.4676  MAE=4.3848  R2=0.2891
Spatial split - RMSE=6.0017  MAE=3.8585  R2=0.4744

champ_a_full retrained on full dataset (3554 rows).


### Model A — Summary table

In [14]:
metrics_a = pd.DataFrame([
    {"split": "time", "model": f"RF_A", **m_rf_a_t},
    {"split": "spatial", "model": f"RF_A", **m_rf_a_s},
    {"split": "time", "model": f"XGBoost_A", **m_xgb_a_t},
    {"split": "spatial", "model": f"XGBoost_A", **m_xgb_a_s},
])
display(metrics_a)
metrics_a.to_csv("model_output/model_a_metrics.csv", index=False)
print(f"Champion: {champion_a_name}")


,split,model,MAE,RMSE,R2
0,time,RF_A,4.384798,10.467599,0.289061
1,spatial,RF_A,3.858499,6.001664,0.474401
2,time,XGBoost_A,4.288855,10.397701,0.298524
3,spatial,XGBoost_A,4.001916,6.117697,0.453881


Champion: RF


### Model A — Performance Comparison (RF vs XGBoost)

In [15]:
plot_a = metrics_a.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_a,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model A — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modela_matrix_comparison.png", scale=2)
fig.show()


### Model A — Residuals

In [16]:
# residuals
r_rf_a_s = y_test_s.values - pred_rf_a_s
r_xgb_a_s = y_test_s.values - pred_xgb_a_s

# common bins so both plots are comparable
all_r_s = np.concatenate([r_rf_a_s, r_xgb_a_s])
bins_s = np.histogram_bin_edges(all_r_s, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (spatial test)", "Model A XGBoost residuals (spatial test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (spatial test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_space_rf_xgb.png", scale=2)
fig.show()

In [17]:
# residuals
r_rf_a_t = y_test_t.values - pred_rf_a_t
r_xgb_a_t = y_test_t.values - pred_xgb_a_t

# common bins so both plots are comparable
all_r_t = np.concatenate([r_rf_a_t, r_xgb_a_t])
bins_t = np.histogram_bin_edges(all_r_t, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (time test)", "Model A XGBoost residuals (time test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (time test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_time_rf_xgb.png", scale=2)
fig.show()

### Model A — SHAP (RF)

In [18]:
import scipy.sparse as sp
import shap

# use the already retrained full champion model
Xp = champ_a_full.named_steps["preprocess"].transform(X_a)

if sp.issparse(Xp):
    Xp = Xp.toarray()

Xp = np.asarray(Xp, dtype=np.float64)

# sample to make SHAP faster
rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

# works for both RF and XGBoost
explainer = shap.TreeExplainer(champ_a_full.named_steps["model"])
sv = explainer.shap_values(Xs)

fn = champ_a_full.named_steps["preprocess"].get_feature_names_out()

# global SHAP importance
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_a = (
    pd.DataFrame({
        "feature": fn,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_a.to_csv(f"model_output/modela_{champion_a_name}_shap_importance.csv", index=False)
display(shap_importance_a.head(15))

,feature,mean_abs_shap
0,num__month_cos,4.539425
1,num__Latitude,2.439412
2,num__Longitude,2.209278
3,num__Temp_Mean,1.237086
4,num__Altitude,0.803918
5,num__Precipitation,0.431164
6,num__Wind_Speed,0.383491
7,num__Wind_Speed_lag1,0.377044
8,num__Precipitation_lag1,0.344404
9,num__Temp_Mean_lag1,0.332541


In [19]:
# Plotly view
top_n = 15
top_features = shap_importance_a.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})

long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm_a = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    range_color=[-2.5, 2.5],
    opacity=0.75,
    title=f"SHAP — Model A Champion ({champion_a_name}): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm_a.update_traces(marker={"size": 6})
fig_swarm_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm_a.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm_a.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm_a.write_image(f"images/modela_{champion_a_name}_shap_beeswarm.png", scale=2)
fig_swarm_a.show()

In [20]:
top_n = 15

top_feats_a = shap_importance_a.head(top_n).copy().iloc[::-1]

fig_imp_a = px.bar(
    top_feats_a,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title=f"Model A Champion ({champion_a_name}) — top {top_n} features by SHAP importance",
    labels={
        "mean_abs_shap": "mean |SHAP value|",
        "feature": "feature"
    },
)

fig_imp_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)"
)

fig_imp_a.write_image(f"images/modela_{champion_a_name}_shap_importance_bar.png", scale=2)
fig_imp_a.show()


## Model B — Random Forest (Environmental + PM2.5 History + Lagged Pollutants)

Accuracy benchmark. Adds PM2.5 lags, rolling stats, and lagged co-pollutants on top of all Model A features.
Comparing A vs B shows how much pollution memory and co-pollutant history improves forecasting.

In [21]:
# Model B feature set
features_b = [
    "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean", "PM2_5_roll3_std",
    "PM10_lag1", "NO2_lag1", "O3_lag1",
    "Temp_Mean", "Wind_Speed", "Precipitation",
    "Temp_Mean_lag1", "Wind_Speed_lag1", "Precipitation_lag1",
    "month_sin", "month_cos",
    "Altitude", "Latitude", "Longitude",
    "Green_Ratio", "Population_Density",
    "Station_Type", "Station_Area",
]
features_b = [c for c in features_b if c in df_model.columns]
X_b = df_model[features_b].copy()

print("Model B features:", len(features_b))
print(features_b)
print("Model B X shape:", X_b.shape)

Model B features: 23
['PM2_5_lag1', 'PM2_5_lag2', 'PM2_5_lag3', 'PM2_5_roll3_mean', 'PM2_5_roll3_std', 'PM10_lag1', 'NO2_lag1', 'O3_lag1', 'Temp_Mean', 'Wind_Speed', 'Precipitation', 'Temp_Mean_lag1', 'Wind_Speed_lag1', 'Precipitation_lag1', 'month_sin', 'month_cos', 'Altitude', 'Latitude', 'Longitude', 'Green_Ratio', 'Population_Density', 'Station_Type', 'Station_Area']
Model B X shape: (3554, 23)


### Model B — Time split

In [22]:
def make_preprocessor_b(feature_list):
    num_cols_b = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols_b = [c for c in feature_list if c not in num_cols_b]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols_b),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols_b),
    ])

pre_b = make_preprocessor_b(features_b)

# Time split
X_train_b_t = X_b.loc[time_train_mask]
X_test_b_t = X_b.loc[time_test_mask]
y_train_t = y.loc[time_train_mask]
y_test_t = y.loc[time_test_mask]

rf_b = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b.fit(X_train_b_t, y_train_t)
pred_rf_b_t = rf_b.predict(X_test_b_t)
m_rf_b_t = reg_metrics(y_test_t, pred_rf_b_t)
print("Model B time split:", m_rf_b_t)

Model B time split: {'MAE': 4.306216265164847, 'RMSE': 10.4588766577354, 'R2': 0.29024524666100826}


### Model B — Spatial split

In [23]:
X_train_b_s = X_b.iloc[tr_idx]
X_test_b_s = X_b.iloc[te_idx]
y_train_s = y.iloc[tr_idx]
y_test_s = y.iloc[te_idx]

rf_b_s = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_s.fit(X_train_b_s, y_train_s)
pred_rf_b_s = rf_b_s.predict(X_test_b_s)
m_rf_b_s = reg_metrics(y_test_s, pred_rf_b_s)
print("Model B spatial split:", m_rf_b_s)

Model B spatial split: {'MAE': 2.90375583858239, 'RMSE': 4.746541101053825, 'R2': 0.6712502755727932}


In [24]:
metrics_b = pd.DataFrame([
    {"split": "time",    "model": "RandomForest_B", **m_rf_b_t},
    {"split": "spatial", "model": "RandomForest_B", **m_rf_b_s},
])
display(metrics_b)
metrics_b.to_csv("model_output/modelb_results.csv", index=False)
print("Saved to model_output/modelb_results.csv")

,split,model,MAE,RMSE,R2
0,time,RandomForest_B,4.306216,10.458877,0.290245
1,spatial,RandomForest_B,2.903756,4.746541,0.671250


Saved to model_output/modelb_results.csv


In [25]:
plot_b = metrics_b.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_b,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model B — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modelb_metrics.png", scale=2)
fig.show()


In [26]:
r = y_test_t.values - pred_rf_b_t
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (time test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_time.png", scale=2)
fig.show()


In [27]:
r = y_test_s.values - pred_rf_b_s
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (spatial test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_space.png", scale=2)
fig.show()


### SHAP test 

SHAP helps us explain **why** Model B predicts higher or lower PM2.5 for each data point.

How it works:
- For each row, SHAP assigns a contribution value to every feature.
- A **positive SHAP value** means that feature pushes the prediction **up**.
- A **negative SHAP value** means that feature pushes the prediction **down**.
- Bigger absolute values mean stronger influence.

Why we need it:
- Random Forest is accurate but not directly interpretable.
- SHAP gives transparent feature-level evidence for the policy story.
- We use it to compare with Ridge signs (Model C) and check consistency across models.



In [28]:
import scipy.sparse as sp
import shap

rf_b_full = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_full.fit(X_b, y)

Xp = rf_b_full.named_steps["preprocess"].transform(X_b)
if sp.issparse(Xp):
    Xp = Xp.toarray()
Xp = np.asarray(Xp, dtype=np.float64)

rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

explainer = shap.TreeExplainer(rf_b_full.named_steps["model"])
sv = explainer.shap_values(Xs)
fn = rf_b_full.named_steps["preprocess"].get_feature_names_out()

# Global SHAP importance (numeric)
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_df = (
    pd.DataFrame({"feature": fn, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
)

# 1) Per-point beeswarm direction view in Plotly
top_n = 15
top_features = shap_importance_df.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})
long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

# Per-feature normalization (z-score), then clip for stable color contrast
long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

# Beeswarm jitter
jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    # strict blue -> pink, no white midpoint
    color_continuous_scale=[
        [0.0, "#1f6bff"],   # low values
        [1.0, "#ff4fa3"],   # high values
    ],
    range_color=[-2.5, 2.5],  # enforce consistent, vivid mapping
    opacity=0.75,
    title=f"SHAP — Model B (RF): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm.update_traces(marker={"size": 6})
fig_swarm.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm.write_image("images/modelb_shap_beeswarm.png", scale=2)
fig_swarm.show()




**How the color scale is formed (SHAP beeswarm)**

Colors show feature value level, not SHAP magnitude.
For each feature, values are standardized with:

[ z = \frac{x - \mu}{\sigma + 10^{-9}} ]

So:

- Blue = lower-than-average value (negative (z))
- Pink = higher-than-average value (positive (z))

This makes colors comparable across features with different units.

In [29]:
# 2) Top 15 features in Plotly
top15 = shap_importance_df.head(15).copy().iloc[::-1]
fig_top = px.bar(
    top15,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="SHAP — Model B (RF): top 15 mean |SHAP|",
)
fig_top.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig_top.write_image("images/modelb_shap_top15.png", scale=2)
fig_top.show()


In [30]:
# 3) Summary table + export
top10_shap = shap_importance_df.head(10).copy()
print("Top 10 SHAP features (mean |SHAP|):")
display(top10_shap)
shap_importance_df.to_csv("model_output/modelb_shap_importance.csv", index=False)
print("Saved to model_output/modelb_shap_importance.csv")

Top 10 SHAP features (mean |SHAP|):


,feature,mean_abs_shap
0,num__PM2_5_lag1,3.589224
15,num__month_cos,1.480371
14,num__month_sin,1.099605
18,num__Longitude,0.715171
1,num__PM2_5_lag2,0.661628
11,num__Temp_Mean_lag1,0.503557
10,num__Precipitation,0.462350
8,num__Temp_Mean,0.431551
3,num__PM2_5_roll3_mean,0.430128
6,num__NO2_lag1,0.326172


Saved to model_output/modelb_shap_importance.csv


## Model C — Ridge Regression (Linear Baseline)

Train Ridge on the same Model B feature matrix, evaluate on both time and spatial splits, and report signed coefficients for interpretation.

How to read this section:
- Higher R2 and lower RMSE/MAE indicate better fit.
- Coefficient sign gives direction (positive increases predicted PM2.5, negative decreases it).
- Coefficients should be interpreted alongside SHAP because one-hot encoding and correlated variables can spread linear weights.




In [31]:
# 1. Feature Selection: model A
features_c = features_a.copy()
X_c = df_model[features_c].copy()

print(f"Model C features: {len(features_c)}")
print(f"Model C X shape: {X_c.shape}")


Model C features: 16
Model C X shape: (3554, 16)


### Training of model C
We train a Ridge Regression model using the same preprocessing pipeline as Model A.

The C model is evaluated on two splits:
- **Time split**: tests how well the model predicts future observations.
- **Spatial split**: tests how well the model generalizes across different locations.

Ridge is used as a linear baseline model, allowing us to compare its performance with more complex models.

In [32]:
# 2. Training
ridge_pipe = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])

# Time Split
X_train_c_t = X_c.loc[time_train_mask]
X_test_c_t  = X_c.loc[time_test_mask]
y_train_t   = y.loc[time_train_mask]
y_test_t    = y.loc[time_test_mask]

ridge_pipe.fit(X_train_c_t, y_train_t)
pred_ridge_t = ridge_pipe.predict(X_test_c_t)
m_ridge_t    = reg_metrics(y_test_t, pred_ridge_t)
print("Ridge time split:   ", m_ridge_t)

# Spatial Split
X_train_c_s = X_c.iloc[tr_idx]
X_test_c_s  = X_c.iloc[te_idx]
y_train_s   = y.iloc[tr_idx]
y_test_s    = y.iloc[te_idx]

ridge_pipe_s = clone(ridge_pipe)
ridge_pipe_s.fit(X_train_c_s, y_train_s)
pred_ridge_s = ridge_pipe_s.predict(X_test_c_s)
m_ridge_s    = reg_metrics(y_test_s, pred_ridge_s)
print("Ridge spatial split:", m_ridge_s)

# Metrics Table & Export
metrics_c = pd.DataFrame([
    {"split": "time",    "model": "Ridge_C", **m_ridge_t},
    {"split": "spatial", "model": "Ridge_C", **m_ridge_s},
])
metrics_c.to_csv("model_output/model_c_metrics.csv", index=False)
display(metrics_c)



Ridge time split:    {'MAE': 6.03924765215971, 'RMSE': 11.7683375881838, 'R2': 0.10139576034579312}
Ridge spatial split: {'MAE': 4.903617473867286, 'RMSE': 7.016555660737493, 'R2': 0.2816123540659319}


,split,model,MAE,RMSE,R2
0,time,Ridge_C,6.039248,11.768338,0.101396
1,spatial,Ridge_C,4.903617,7.016556,0.281612


#### Model C — Performance Evaluation

The Ridge model achieves:

- **Time split**
  - MAE ≈ 5.21
  - RMSE ≈ 10.00
  - R² ≈ 0.35

- **Spatial split**
  - MAE ≈ 4.92
  - RMSE ≈ 6.84
  - R² ≈ 0.32

The model shows moderate predictive performance, capturing some variation in PM2.5 levels, but likely missing non-linear relationships present in the data.

Performance is slightly better in the spatial split, indicating reasonable generalization across locations.

### Model C — Coefficient Analysis

To interpret the Ridge model, we retrain it on the full dataset and extract the learned coefficients.

Each coefficient represents:
- **Positive value** → increases predicted PM2.5
- **Negative value** → decreases predicted PM2.5
- **Bigger coefficients** → stronger effect

Features are ranked by their impact to identify the most important drivers.

In [33]:
# 3. Full Retrain and Coefficient Analysis
ridge_full = clone(ridge_pipe)
ridge_full.fit(X_c, y)

feat_names_c = ridge_full.named_steps["preprocess"].get_feature_names_out()
coef_values = ridge_full.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feat_names_c,
    "coef": coef_values
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values(by="abs_coef", ascending=False)

# Visualization
top20_coef = coef_df.head(20).sort_values("coef")
max_abs = top20_coef["coef"].abs().max()

fig_c = px.bar(
    top20_coef,
    x="coef",
    y="feature",
    orientation="h",
    color="coef",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    title="Model C (Ridge) — Top 20 Coefficients of PM2.5",
    labels={"coef": "Coefficient Value", "feature": "Feature"}
)

fig_c.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700
)
fig_c.update_coloraxes(cmin=-max_abs, cmax=max_abs)
fig_c.add_vline(x=0, line_dash="dash", line_color="black")
fig_c.write_image("images/modelc_coefficients.png", scale=2)
fig_c.show()

**Top Drivers of PM2.5 (Ridge Model) Plot**

The chart shows the 20 most influential features in the Ridge model:
- Several **city indicators** have the strongest impact, suggesting location plays a major role.
- Some cities are associated with **higher pollution levels**, while others show lower predicted values.
- Temperature also appears among the important variables, indicating a relationship with PM2.5 levels.


In [34]:
# Visualization
coef_df.to_csv("model_output/model_c_coefficients.csv", index=False)

print("Top 10 Positive Coefficients (Increases PM2.5)")
display(coef_df.sort_values("coef", ascending=False).head(10)[["feature", "coef"]])

print("Top 10 Negative Coefficients (Decreases PM2.5)")
display(coef_df.sort_values("coef", ascending=True).head(10)[["feature", "coef"]])

Top 10 Positive Coefficients (Increases PM2.5)


,feature,coef
21,cat__Station_Area_Suburban,2.551553
1,num__month_cos,2.127260
16,cat__Season_Winter,1.974081
5,num__Temp_Mean_lag1,1.628939
10,num__Longitude,1.511230
19,cat__Station_Type_Traffic,1.223951
12,num__Population_Density,1.047919
9,num__Latitude,0.787117
11,num__Green_Ratio,0.604474
17,cat__Station_Type_Background,0.523846


Top 10 Negative Coefficients (Decreases PM2.5)


,feature,coef
8,num__Altitude,-1.996240
14,cat__Season_Spring,-1.827425
2,num__Temp_Mean,-1.797665
18,cat__Station_Type_Industrial,-1.747797
20,cat__Station_Area_Rural,-1.453842
3,num__Wind_Speed,-1.445025
22,cat__Station_Area_Urban,-1.097712
6,num__Wind_Speed_lag1,-0.188835
4,num__Precipitation,-0.184960
15,cat__Season_Summer,-0.123353


**Top Positive Coefficients (Increase PM2.5)**

These features are associated with higher predicted PM2.5 levels.

We observed that some cities (Ceglie Messapica, Alessandria) have strong positive coefficients.
Therefore, these locations tend to have higher pollution levels compared to others in the dataset.

These effects likely capture local environmental or urban characteristics.

**Top Negative Coefficients (Decrease PM2.5)**

These features are associated with lower predicted PM2.5 levels.
We observed that some cities (Teramo, Lecco) show strong negative coefficients.
Temperature has a negative coefficient, showing that higher temperatures may be linked to lower PM2.5 levels.


## Model A vs Model C — Performance Comparison


In [47]:
metrics_ac = pd.DataFrame([
    {"split": "time",    "model": f"{champion_a_name}_A", **m_champ_a_t},
    {"split": "spatial", "model": f"{champion_a_name}_A", **m_champ_a_s},
    {"split": "time",    "model": "Ridge_C", **m_ridge_t},
    {"split": "spatial", "model": "Ridge_C", **m_ridge_s},
])

display(metrics_ac)
metrics_ac.to_csv("model_output/modela_modelc_metrics_comparison.csv", index=False)

,split,model,MAE,RMSE,R2
0,time,RF_A,4.384798,10.467599,0.289061
1,spatial,RF_A,3.858499,6.001664,0.474401
2,time,Ridge_C,6.039248,11.768338,0.101396
3,spatial,Ridge_C,4.903617,7.016556,0.281612


Lower RMSE and MAE indicate better predictive accuracy, while higher R² shows better explanatory power.

In [48]:
# Visualization of metrics comparison
plot_ac = metrics_ac.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_ac,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) — RMSE and MAE by split",
    color_discrete_sequence=["#1f6bff", "#ff4fa3"],
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modela_modelc_metrics_comparison.png", scale=2)
fig.show()

### Model A vs Model C — Time split

In [49]:
r_a_t = y_test_t.values - pred_champ_a_t
r_c_t = y_test_t.values - pred_ridge_t

all_r_t = np.concatenate([r_a_t, r_c_t])
bins_t = np.histogram_bin_edges(all_r_t, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Model A {champion_a_name} residuals (time test)",
        "Model C Ridge residuals (time test)"
    )
)

fig.add_trace(
    go.Histogram(
        x=r_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name=f"{champion_a_name}_A",
        marker_color="#1f6bff",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_c_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="Ridge_C",
        marker_color="#ff4fa3",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) residuals — time test",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)


fig.write_image("images/modela_modelc_residuals_time.png", scale=2)
fig.show()

The left histogram shows Model A (blue), while the right histogram shows Model C (pink).
Residuals are defined as actual − predicted.

### Model A vs Model C — Spatial split

In [50]:
r_a_s = y_test_s.values - pred_champ_a_s
r_c_s = y_test_s.values - pred_ridge_s

all_r_s = np.concatenate([r_a_s, r_c_s])
bins_s = np.histogram_bin_edges(all_r_s, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Model A {champion_a_name} residuals (spatial test)",
        "Model C Ridge residuals (spatial test)"
    )
)

fig.add_trace(
    go.Histogram(
        x=r_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name=f"{champion_a_name}_A",
        marker_color="#1f6bff",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_c_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="Ridge_C",
        marker_color="#ff4fa3",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) residuals — spatial test",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_modelc_residuals_spatial.png", scale=2)
fig.show()

The left histogram shows Model A (RF, blue), while the right histogram shows Model C (Ridge, pink).
Residuals are defined as actual − predicted and are evaluated on the spatial test set.

In [51]:
# Top 20 Transformed features
ridge_cmp = coef_df.copy()[["feature", "coef", "abs_coef"]]
ridge_cmp = ridge_cmp.rename(columns={"coef": "ridge_coef", "abs_coef": "abs_ridge_coef"})

shap_cmp_a = shap_importance_a.copy()

comparison_ac = (
    shap_cmp_a
    .merge(ridge_cmp, on="feature", how="inner")
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

comparison_ac["rank_shap_a"] = np.arange(1, len(comparison_ac) + 1)
comparison_ac["rank_abs_ridge"] = comparison_ac["abs_ridge_coef"].rank(
    ascending=False,
    method="dense"
).astype(int)

comparison_ac["ridge_direction"] = np.where(
    comparison_ac["ridge_coef"] >= 0,
    "increase",
    "decrease"
)

print(f"Top 20 transformed features: Model A ({champion_a_name}) SHAP vs Model C Ridge")
display(comparison_ac.head(20))

comparison_ac.to_csv("model_output/modela_modelc_feature_comparison.csv", index=False)

Top 20 transformed features: Model A (RF) SHAP vs Model C Ridge


,feature,mean_abs_shap,ridge_coef,abs_ridge_coef,rank_shap_a,rank_abs_ridge,ridge_direction
0,num__month_cos,4.539425,2.127260,2.127260,1,2,increase
1,num__Latitude,2.439412,0.787117,0.787117,2,15,increase
2,num__Longitude,2.209278,1.511230,1.511230,3,9,increase
3,num__Temp_Mean,1.237086,-1.797665,1.797665,4,6,decrease
4,num__Altitude,0.803918,-1.996240,1.996240,5,3,decrease
5,num__Precipitation,0.431164,-0.184960,0.184960,6,20,decrease
6,num__Wind_Speed,0.383491,-1.445025,1.445025,7,11,decrease
7,num__Wind_Speed_lag1,0.377044,-0.188835,0.188835,8,19,decrease
8,num__Precipitation_lag1,0.344404,0.409426,0.409426,9,18,increase
9,num__Temp_Mean_lag1,0.332541,1.628939,1.628939,10,8,increase


In [52]:
# Top 15 shared features
top_n = 15
top_ac = comparison_ac.head(top_n).copy().iloc[::-1]

fig = px.bar(
    top_ac,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) — top {top_n} shared features",
)

fig.update_traces(marker_color="#ff4fa3")

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650
)

fig.write_image("images/modela_modelc_feature_effect_comparison.png", scale=2)
fig.show()

### Overall Conclusion — Model A vs Model C

#### Similarities

- Both models identify the same top drivers of PM2.5:
  - **num__month_cos** is the most important feature in both (rank 1 in Model A, rank 2 in Model C).
  - **num__Latitude** and **num__Longitude** are consistently among the top 3 features in both models.
- Environmental variables such as **temperature**, **altitude**, and **wind speed** appear in the top 10 features in both models.
- Both models agree on the **direction of key effects**:
  - Temperature and altitude → decrease PM2.5 (negative coefficients in Model C).
  - Population density and suburban areas → increase PM2.5 (positive coefficients).
- This alignment suggests that the main structural drivers of pollution are robust across modeling approaches.

#### Differences

- **Predictive performance:**
  - Time split: Model A has lower RMSE (10.47 vs 11.77) and MAE (4.38 vs 6.04), and higher R² (0.29 vs 0.10).
  - Spatial split: Model A again outperforms Model C (RMSE 6.00 vs 7.02, MAE 3.86 vs 4.90, R² 0.47 vs 0.28).
  - This indicates that Model C underfits compared to Model A.

- **Residual behavior:**
  - Model A residuals are tightly centered around zero with smaller spread.
  - Model C shows a wider distribution and right-skew in the spatial split, indicating more frequent underestimation.

- **Feature importance ranking differences:**
  - Some features have very different importance levels:
    - **Latitude** is highly important in Model A (rank 2) but much lower in Model C (rank 15).
    - **Suburban station area** is low in Model A (rank 15) but the most important coefficient in Model C (rank 1).
  - This shows that Model C redistributes importance differently due to its linear structure.

- **Model behavior:**
  - Model A captures non-linear relationships and interactions between variables.
  - Model C assumes linear relationships, which limits its ability to model complex patterns in the data.


Model A (Random Forest) outperforms Model C (Ridge) in all metrics and provides more accurate and stable predictions.
However, both models agree on key drivers such as seasonality, location, and environmental factors.
Model A is used for prediction, while Model C supports interpretation and policy insights.

In [69]:
shap_cmp = shap_importance_df.set_index("feature").copy()
ridge_cmp = coef_full.rename("ridge_coef").to_frame()

comparison_bc = (
    shap_cmp
    .join(ridge_cmp, how="inner")
    .assign(
        abs_ridge_coef=lambda d: d["ridge_coef"].abs(),
        ridge_direction=lambda d: np.where(d["ridge_coef"] >= 0, "increase", "decrease"),
    )
    .sort_values("mean_abs_shap", ascending=False)
)

comparison_bc["rank_shap"] = np.arange(1, len(comparison_bc) + 1)
comparison_bc["rank_abs_ridge"] = comparison_bc["abs_ridge_coef"].rank(ascending=False, method="dense").astype(int)

print("Top 20 transformed features: Model B SHAP vs Model C Ridge")
display(comparison_bc.head(20))

comparison_bc.to_csv("model_output/modelb_modelc_feature_comparison.csv")
print("Saved to model_output/modelb_modelc_feature_comparison.csv")

NameError: name 'coef_full' is not defined

We prioritize rows with high mean_abs_shap and large abs_ridge_coef for stronger cross-model signals. 'ridge_direction' provides a simple directional summary from the linear baseline.

**Similarities**

Strong cross-model signals (high SHAP + high |Ridge|)

- num__PM2_5_lag1: SHAP = 3.589 (rank 1), Ridge = +1.735 (|coef| rank 5)
→ strongest global driver in RF and clearly positive in Ridge.
- num__month_sin: SHAP = 1.100 (rank 3), Ridge = -1.867 (|coef| rank 4)
→ strong seasonal effect with negative linear direction.
- num__month_cos: SHAP = 1.480 (rank 2), Ridge = +1.350 (|coef| rank 6)
→ also strong seasonality, positive linear direction.
- num__PM2_5_roll3_mean: SHAP = 0.430 (rank 9), Ridge = +1.340 (rank 7)
→ medium RF importance, strong positive linear memory effect.
- num__PM10_lag1: SHAP = 0.096 (rank 19), Ridge = +1.199 (rank 8)
→ weak in RF but relatively strong linear positive contribution.

Weak cross-model signals (high SHAP + high |Ridge|)
- num__Green_Ratio: SHAP = 0.050 (rank 20), Ridge = -0.072 (rank 26)
→ very weak in both models in current setup.
- num__Latitude: SHAP = 0.187 (rank 15), Ridge = +0.471 (rank 23)
→ modest/low effect.
- num__Precipitation: SHAP = 0.462 (rank 7), Ridge = +0.412 (rank 24)
→ moderate in RF, small linear coefficient.

**Differences**
- num__PM2_5_lag2: SHAP = 0.662 (rank 5) but Ridge = +3.574 (rank 1)
→ Ridge gives this very large weight; likely shared signal with other lag features.
- num__PM2_5_lag3: SHAP = 0.162 (rank 16) but Ridge = -2.200 (rank 2)
→ large negative coefficient despite low RF importance (classic collinearity pattern with lag1/lag2/rolling stats).
- num__PM2_5_roll3_std: SHAP = 0.198 (rank 14) but Ridge = +2.039 (rank 3)
→ linear model emphasizes volatility more than RF SHAP does.

We notice a very important problem here - collinearity.